In [ ]:
!mkdir -p data

In [ ]:
import requests

def extract_data(url, filename):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }
    try:
        response = requests.get(url, headers=headers, stream=True)
        response.raise_for_status() # Garante que erros HTTP disparem uma exceção
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Sucesso: {filename} baixado.")
    except Exception as e:
        print(f"Erro ao baixar {filename}: {e}")

In [ ]:
extract_data("https://data.boston.gov/dataset/8048697b-ad64-4bfc-b090-ee00169f2323/resource/c9509ab4-6f6d-4b97-979a-0cf2a10c922b/download/lagan_311_open_data_2015_rvsd3.csv", "data/dados_2015.csv")
extract_data("https://data.boston.gov/dataset/8048697b-ad64-4bfc-b090-ee00169f2323/resource/b7ea6b1b-3ca4-4c5b-9713-6dc1db52379a/download/lagan_311_open_data_2016_rvsd3.csv", "data/dados_2016.csv")
extract_data("https://data.boston.gov/dataset/8048697b-ad64-4bfc-b090-ee00169f2323/resource/30022137-709d-465e-baae-ca155b51927d/download/lagan_311_open_data_2017_rvsd3.csv", "data/dados_2017.csv")
extract_data("https://data.boston.gov/dataset/8048697b-ad64-4bfc-b090-ee00169f2323/resource/2be28d90-3a90-4af1-a3f6-f28c1e25880a/download/lagan_311_open_data_2018_rvsd3.csv", "data/dados_2018.csv")
extract_data("https://data.boston.gov/dataset/8048697b-ad64-4bfc-b090-ee00169f2323/resource/ea2e4696-4a2d-429c-9807-d02eb92e0222/download/lagan_311_open_data_2019_rvsd3.csv", "data/dados_2019.csv")
extract_data("https://data.boston.gov/dataset/8048697b-ad64-4bfc-b090-ee00169f2323/resource/6ff6a6fd-3141-4440-a880-6f60a37fe789/download/lagan_311_open_data_2020_rvsd3.csv", "data/dados_2020.csv")

In [ ]:
arquivos = [

    "data/dados_2015.csv",

    "data/dados_2016.csv",

    "data/dados_2017.csv",

    "data/dados_2018.csv",

    "data/dados_2019.csv",

    "data/dados_2020.csv",

]

In [ ]:
dfs = {}

In [ ]:
import pandas as pd

for arquivo in arquivos:
    ano = arquivo.split("_")[-1].split(".")[0]
    dfs[ano] = pd.read_csv(arquivo)

In [ ]:
dfs["2018"].head()

In [ ]:
pip install boto3

In [ ]:
import boto3
from google.colab import userdata

aws_access_key_id = userdata.get("AWS_ACCESS_KEY_ID")
aws_secret_access_key = userdata.get("AWS_SECRET_ACCESS_KEY")

s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name="us-east-1"
)

In [ ]:
contet ="""Olá, S3 """

with open("hello-s3.txt", "w") as f:
  f.write(contet)

In [ ]:
s3.upload_file("hello-s3.txt", "datalake-aws-project", "bronze/hello-s3" )

In [ ]:
from io import BytesIO

for ano, df in dfs.items():
    # Copia o DataFrame para evitar alterar o original na memória de forma inesperada
    df_temp = df.copy()

    # Converte a coluna 'precinct' para string, lidando com possíveis valores nulos
    if 'precinct' in df_temp.columns:
        df_temp['precinct'] = df_temp['precinct'].astype(str)

    parquet_buffer = BytesIO()
    df_temp.to_parquet(parquet_buffer)

    s3.put_object(
        Bucket="datalake-aws-project",
        Key=f"bronze/dados_{ano}.parquet",
        Body=parquet_buffer.getvalue(),
    )

In [ ]:
response = s3.list_objects(Bucket="datalake-aws-project")

In [ ]:
keys = [obj["Key"] for obj in response["Contents"]]
print(keys)

In [ ]:
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name="us-east-1"
)

identidade = sts.get_caller_identity()

print("Conexão com AWS realizada com sucesso!")
print("Conta AWS:", identidade["Account"])

